In [20]:
from utils import *
import matplotlib.pyplot as plt
import numpy as np
import torch
from pprint import pprint

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
def analyze_model_path(path):
    experiment_output = {}
    model, env, metrics_data = load_model(path)  # path to the model checkpoint

    if metrics_data['args'].get('fc1_encoder_enabled', False): model_type = "cnn"
    elif metrics_data['args'].get('fc1_activation', 'leaky_relu') == 'leaky_relu': model_type = "mlp"
    else: model_type = "linear model"

    print(f"Analyzing model of type: {model_type}")
    batch_size = metrics_data['args'].get('batch_size', 2048)
    hidden_size = metrics_data['args'].get('fc1_hidden', 25)
    split_vector = metrics_data['args'].get('split_vector', False)
    filter_channels = metrics_data['args'].get('fc1_filter_channels', [8, 16, 32])
    filter_sizes = metrics_data['args'].get('fc1_filter_sizes', [3, 3, 3])
    mag_cost = metrics_data['args'].get('mag_cost', 1.0)
    experiment_output["batch size"] = batch_size
    experiment_output["hidden size"] = hidden_size
    experiment_output["split vector"] = split_vector
    experiment_output["filter channels"] = filter_channels
    experiment_output["filter sizes"] = filter_sizes
    experiment_output["mag cost"] = mag_cost
    experiment_output["model type"] = model_type

    train_history, val_history = get_model_history(metrics_data)
    experiment_output["min val loss"] = np.min(val_history)
    experiment_output["final val loss"] = val_history[-1]
    experiment_output[" train loss"] = np.min(train_history)
    experiment_output["final train loss"] = train_history[-1]

    noise_levels = np.logspace(-8, -5.5, num=7)
    noise_levels = np.concatenate(([1e-7, 1e-6], noise_levels))

    evaluation_stats1, predictions1, truths1 = test_noise_levels(model, env, metrics_data, noise_levels=noise_levels, noise_mode_amounts=[25], repetitions=5, loops=1, delta_t=1e-1)
    experiment_output["exp1 evaluation stats"] = evaluation_stats1

    evaluation_stats2, predictions2, truths2 = test_noise_levels(model, env, metrics_data, noise_levels=[metrics_data['dataset_meta']['dataset_config']['dm_random_noise']], noise_mode_amounts=[10, 25, 50, 100, 500], repetitions=5, loops=1, delta_t=1e-1)
    experiment_output["exp2 evaluation stats"] = evaluation_stats2

    return experiment_output


In [50]:
import os 

folders_to_analyze = ['mlp orth']

prefix_path = r"a:\Projects\DM + RL\RL\jobs"
all_analysis_results = {}
for folder in folders_to_analyze:
    folder_path = os.path.join(prefix_path, folder)
    for experiment in os.listdir(folder_path):
        experiment_path = os.path.join(folder_path, experiment)
        if os.path.isdir(experiment_path):
            print(f"Analyzing experiment at: {experiment_path}")
            analysis_results = analyze_model_path(experiment_path, )
            all_analysis_results[experiment] = analysis_results


Analyzing experiment at: a:\Projects\DM + RL\RL\jobs\mlp orth\exp-fc1-zscore-ep1500-bs256-lr1e-3-s42-h128-64-images
initializing coronagraph env. might take a minute.
Analyzing model of type: mlp
env.num_noise_modes: 500
Using split vector: False


noise sweep: 100%|██████████| 9/9 [00:02<00:00,  3.67it/s]


env.num_noise_modes: 500
Using split vector: False


noise sweep: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]


Analyzing experiment at: a:\Projects\DM + RL\RL\jobs\mlp orth\exp-fc1-zscore-ep1500-bs256-lr1e-3-s42-h128-64-images-do0.1
initializing coronagraph env. might take a minute.
Analyzing model of type: mlp
env.num_noise_modes: 500
Using split vector: False


noise sweep:  33%|███▎      | 3/9 [00:01<00:02,  2.69it/s]


KeyboardInterrupt: 

In [51]:
import json

pprint("Final analysis results:")
pprint(all_analysis_results)

def _json_default(obj):
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

output_path = os.path.join(prefix_path, "all_analysis_results.json")
with open(output_path, "w") as f:
    json.dump(all_analysis_results, f, default=_json_default, indent=2)
print(f"Saved JSON to {output_path}")

'Final analysis results:'
{'exp-fc1-zscore-ep1500-bs256-lr1e-3-s42-h128-64-images': {' train loss': np.float64(0.026115206399559974),
                                                           'batch size': 256,
                                                           'exp1 evaluation stats': {'cosine_similarity': {'mean': [[-0.9789211081775303],
                                                                                                                    [0.017063814954554758],
                                                                                                                    [-0.7721962752663862],
                                                                                                                    [-0.8908381773654309],
                                                                                                                    [-0.9553811006033964],
                                                                                            